# World Model Trainer

> Train a LeWM-like actio-conditioned model.

In [ ]:
#| default_exp trainers.world_model

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math
import torch
import os
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.utils as vutils
import wandb
import hydra
from pathlib import Path
from fastcore.utils import patch
from loguru import logger
from omegaconf import DictConfig
from einops import rearrange
import torch.nn.functional as F
from c3jepa_wm.utils.checkpointer import RetrospectiveCheckpointer


In [ ]:
#| export
class TrainerScheduler:
    def __init__(self, wm_optimizer):
        self.wm_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            wm_optimizer, mode='min', factor=0.5, patience=5
        )

    def step(self, val_loss):
        self.wm_scheduler.step(val_loss['val_jepa_loss'])
        
    

In [ ]:
#| export
class BaseTrainer:
    def __init__(self, 
                 data_module, 
                 device, 
                 slurm_jobid= None, 
                 wm_lr=1e-4, 
                 epochs=100, 
                 project_name="c3jepa_wm",
                 ckp_dir= "checkpoints",
                 save_dir= "outputs"):
        
        self.data_module = data_module
        
        self.device = device
        
        self.data_module.setup()
        self.train_loader = self.data_module.train_dataloader()
        self.val_loader = self.data_module.val_dataloader()

        self.slurm_jobid = slurm_jobid if slurm_jobid else "default_job"
        self.wm_lr = wm_lr
        self.epochs = epochs
        self.project_name = project_name
        self.ckp_dir = ckp_dir
        self.save_dir = save_dir

    def init_optimizer(self, model, lr, weight_decay=0.01):
        return torch.optim.AdamW(
                list(model.parameters()),
                lr=lr,
                weight_decay=weight_decay
            )
    
    def train_epoch(self, epoch):
        raise NotImplementedError("train_epoch method must be implemented by subclasses.")   

    def validate_epoch(self, epoch):
        raise NotImplementedError("validate method must be implemented by subclasses.")
    

In [ ]:
#| export
class WMTrainer(BaseTrainer):
    def __init__(self, data_module, model, device, slurm_jobid, wm_lr,
                 history_size, num_preds, lambda_sigreg, **kwargs):
        super().__init__(
            data_module=data_module, device=device, wm_lr=wm_lr,
            slurm_jobid=slurm_jobid, **kwargs
        )
        self.history_size = history_size
        self.num_preds = num_preds
        self.lambda_sigreg = lambda_sigreg
        self.model = model["jepa"].to(device)
        self.wm_optimizer = self.init_optimizer(self.model, lr=self.wm_lr, weight_decay=1e-3)
        self.scheduler = TrainerScheduler(self.wm_optimizer)

        self.save_dir = Path(hydra.utils.to_absolute_path(self.save_dir))
        self.save_dir.mkdir(exist_ok=True, parents=True)

        self.ck_pointer = RetrospectiveCheckpointer(
            project_name=self.project_name, ckp_dir=self.ckp_dir,
            slurm_jobid=self.slurm_jobid, agent_id="WM_trainer", rank=0, n_best=3
        )
        


In [ ]:
#| export
@patch
def fit(self: WMTrainer, cfg: DictConfig):
    for epoch in range(1, cfg.pipeline.max_epochs + 1):
        train_loss = self.train_epoch(epoch)
        metrics = self.evaluate_epoch(epoch)
        self.scheduler.step(metrics)
        self.checkpoint(epoch, metrics)
   

In [ ]:
#| export
@patch
def train_epoch(self: WMTrainer, epoch):
    self.model.train()
    total_loss_jepa = 0.0
    for batch_idx, batch in enumerate(self.train_loader):
        batch = {k: v.to(self.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        jepa_loss = self.train_batch(epoch, batch)
        total_loss_jepa += jepa_loss
        logger.info(f"Batch {batch_idx+1}/{len(self.train_loader)} epoch {epoch} loss {jepa_loss:.4f}")

    avg_loss_jepa = total_loss_jepa / len(self.train_loader)
    logger.info(f"Epoch [{epoch}/{self.epochs}] - Train JEPA Loss: {avg_loss_jepa:.4f}")
    return avg_loss_jepa



In [ ]:
#| export
@patch
def train_batch(self: WMTrainer, epoch, batch):
    B = batch["pixels"].shape[0]
    T = self.history_size
    self.wm_optimizer.zero_grad()

    # phase 0: blind pretraining — encode() naturally ignores sender_pov/sender_csi,
    # and predict() is called with msg_indices=None (its default) → action-only conditioning
    output = self.model.encode(batch)
    emb = output["emb"]
    act_emb = output["act_emb"].reshape(B, T, -1)
    ctx_emb = emb[:, :T]
    ctx_act = act_emb[:, :T]
    tgt_emb = emb[:, self.num_preds:]

    pred_emb = self.model.predict(ctx_emb, ctx_act)
    output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, self.lambda_sigreg)

    loss_dict = {k: v for k, v in output.items() if "loss" in k}
    wandb.log({f"train_{k}": v.item() for k, v in loss_dict.items()})

    output['jepa_loss'].backward()
    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
    self.wm_optimizer.step()
    return output['jepa_loss'].item()


In [ ]:
#| export
@patch
@torch.no_grad()
def evaluate_epoch(self: WMTrainer, epoch):
    self.model.eval()
    all_metrics = {}
    for batch in self.val_loader:
        batch = {k: v.to(self.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        metrics = self.evaluate_batch(batch)          # no msg_indices — phase 0 is blind
        for k, v in metrics.items():
            all_metrics.setdefault(k, []).append(v)

    avg_metrics = {k: sum(v) / len(v) for k, v in all_metrics.items()}
    wandb.log({**avg_metrics, "epoch": epoch})
    self.model.train()
    return avg_metrics


In [ ]:
#| export
@patch
@torch.no_grad()
def evaluate_batch(self: WMTrainer, batch):
    B = batch["pixels"].shape[0]
    T = self.history_size

    output = self.model.encode(batch)
    emb = output["emb"]
    act_emb = output["act_emb"].reshape(B, T, -1)
    ctx_emb = emb[:, :T]
    ctx_act = act_emb[:, :T]
    tgt_emb = emb[:, self.num_preds:]

    pred_emb = self.model.predict(ctx_emb, ctx_act)
    output = self.model.loss_fn(output, pred_emb, tgt_emb, emb, self.lambda_sigreg)
    return {"val_jepa_loss": output['jepa_loss'].item()}


In [ ]:
#| export
@patch
def checkpoint(self: WMTrainer, epoch, val_loss):
    checkpoint_state = {
        "epoch": epoch,
        "wm_optimizer_state_dict": self.wm_optimizer.state_dict(),
        "wm_model_state_dict": self.model.state_dict(),
        "val_jepa_loss": val_loss['val_jepa_loss'],
    }
    acc = -val_loss["val_jepa_loss"]   # always tracked this way — no phase gate needed here
    self.ck_pointer.save_checkpoint(state=checkpoint_state, current_acc=acc, step=epoch)
    

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()